# Lab Task 3 — Forward and Backward Propagation

**Source:** *Understanding Deep Learning*, Laboratory Task 3.

**Instruction:** Use the exact values from Task 2, apply ReLU, and perform forward **and backward** propagation using NumPy with learning rate \(0.001\).

For a single scalar output, use the **half-squared-error** loss

\[
L=\frac{1}{2}(\hat y-y)^2.
\]

With ReLU \(f'(z)=1\) when \(z>0\) and \(0\) otherwise, the gradients are

\[
\delta_o = (\hat y-y)f'(z_o),\quad
\delta_h = (\delta_o W_o)\odot f'(z_h).
\]

This is a manual NumPy implementation (no automatic gradients). All gradients are calculated before weights are updated.


In [1]:
import numpy as np

x = np.array([1., 0., 1.])
y = np.array([1.])
lr = 0.001
W_h = np.array([[0.2, -0.3], [0.4, 0.1], [-0.5, 0.2]], dtype=float)
b_h = np.array([-0.4, 0.2], dtype=float)
W_o = np.array([[-0.3], [-0.2]], dtype=float)
b_o = np.array([0.1], dtype=float)

relu = lambda z: np.maximum(0, z)
relu_derivative = lambda z: (z > 0).astype(float)

# Forward pass
z_h = x @ W_h + b_h
h = relu(z_h)
z_o = h @ W_o + b_o
y_pred_before = relu(z_o)
loss_before = float(0.5 * np.sum((y_pred_before - y)**2))

# Backpropagation for L = 0.5 * (prediction - target)^2
delta_o = (y_pred_before - y) * relu_derivative(z_o)
grad_W_o = np.outer(h, delta_o)
grad_b_o = delta_o.copy()
delta_h = (W_o @ delta_o) * relu_derivative(z_h)
grad_W_h = np.outer(x, delta_h)
grad_b_h = delta_h.copy()

# Gradient descent update
W_o = W_o - lr * grad_W_o
b_o = b_o - lr * grad_b_o
W_h = W_h - lr * grad_W_h
b_h = b_h - lr * grad_b_h

# Check the change using a NEW forward pass
z_h_new = x @ W_h + b_h
h_new = relu(z_h_new)
y_pred_after = relu(h_new @ W_o + b_o)
loss_after = float(0.5 * np.sum((y_pred_after - y)**2))

print('Before: prediction =', y_pred_before, 'loss =', round(loss_before, 8))
print('Output delta:', delta_o)
print('Hidden delta:', delta_h)
print('Updated input-to-hidden weights:\n', W_h)
print('Updated hidden-to-output weights:\n', W_o)
print('Updated hidden biases:', b_h)
print('Updated output bias:', b_o)
print('After: prediction =', y_pred_after, 'loss =', round(loss_after, 8))

assert np.isclose(y_pred_before.item(), 0.08)
assert np.isclose(loss_before, 0.4232)
assert loss_after < loss_before, 'The first gradient step should reduce the loss.'


Before: prediction = [0.08] loss = 0.4232
Output delta: [-0.92]
Hidden delta: [0.    0.184]
Updated input-to-hidden weights:
 [[ 0.2      -0.300184]
 [ 0.4       0.1     ]
 [-0.5       0.199816]]
Updated hidden-to-output weights:
 [[-0.3     ]
 [-0.199908]]
Updated hidden biases: [-0.4       0.199816]
Updated output bias: [0.10092]
After: prediction = [0.08103955] loss = 0.42224416


**Interpretation:** The first ReLU hidden neuron receives no gradient because its pre-activation is \(-0.7\). The second neuron and the output unit do receive gradients. After one SGD step the loss decreases slightly, demonstrating how backpropagation adjusts weights and biases toward the target. This activity intentionally performs **one update**, not an entire training run.
